# 05 SafeTune Interpret: the refusal signal

Uses SafeTune's `extract_refusal_direction` to find the direction in the model's activations that separates harmful from harmless prompts. The direction is learned from 40 + 40 English JBB prompts that are not in the test set. Each test prompt's last-token activation is then projected onto it and rescaled so English harmless = 0 and English harmful (plain) = 1.

SafeTune's automatic layer selection runs a generation check that does not fit in T4 memory with Tiny Aya's vocabulary, so `select_directions=False` is used and the layer is chosen by how well it separates the calibration prompts.

Needs a GPU. Input: the labeled CSV.

In [ ]:
!pip uninstall -y -q torchao
!pip install -q safetune
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score
from safetune import steer
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
except ImportError:
    login(token=os.environ.get("HF_TOKEN"))

LABELED =  ""
MODEL_NAME = "CohereLabs/tiny-aya-global"
N_CAL = 40
ATTACKS = ["plain", "villain", "story", "fictional_world"]

df = pd.read_csv(LABELED).fillna("")
df["refusal"] = df["refusal"].astype(str).str.upper() == "TRUE"
df.loc[(df["type"] == "harmful") & (df["label"] == "") & df["refusal"], "label"] = "refused"
test = df[((df["type"] == "benign") & (df["attack"] == "plain")) |
          ((df["type"] == "harmful") & df["attack"].isin(ATTACKS))].copy().reset_index(drop=True)
print("Test prompts to measure:", len(test))

jbb = load_dataset("JailbreakBench/JBB-Behaviors", "behaviors")
hb, bb = jbb["harmful"].to_pandas(), jbb["benign"].to_pandas()
used = set(df["behavior"])
cal_h = hb[~hb["Behavior"].isin(used)].sample(N_CAL, random_state=0)["Goal"].tolist()
cal_b = bb[~bb["Behavior"].isin(used)].sample(N_CAL, random_state=0)["Goal"].tolist()
print("Calibration:", len(cal_h), "harmful +", len(cal_b), "harmless (none from the test set)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="cuda:0")
model.eval()

direction, layer, per_layer = steer.extract_refusal_direction(
    model, tokenizer, cal_h, cal_b, steer.RefusalDirectionConfig(select_directions=False))
dirs = {int(l): (v.float().cpu() / v.float().norm().cpu()) for l, v in (per_layer or {}).items()}
dirs.setdefault(int(layer), direction.float().cpu() / direction.float().norm().cpu())
torch.cuda.empty_cache()
print(f"SafeTune extracted directions for {len(dirs)} layers")

def hidden_states(prompt):
    text = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                         tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model(**ids, output_hidden_states=True)
    return [h[0, -1].float().cpu() for h in out.hidden_states]

Hc_h = [hidden_states(p) for p in cal_h]
Hc_b = [hidden_states(p) for p in cal_b]
n_states = len(Hc_h[0])
best = None
for l in dirs:
    for off in (0, 1):
        if l + off >= n_states:
            continue
        a = np.array([float(h[l + off] @ dirs[l]) for h in Hc_h])
        b = np.array([float(h[l + off] @ dirs[l]) for h in Hc_b])
        sep = (a.mean() - b.mean()) / (np.sqrt((a.var() + b.var()) / 2) + 1e-6)
        if best is None or sep > best[2]:
            best = (l, off, sep)
layer, OFFSET, sep = best
print(f"Best layer: {layer} (separation {sep:.2f} on calibration prompts)")

H = [hidden_states(p) for p in test["prompt"]]
print("Measured", len(H), "test prompts")

en_h = (test["lang"] == "english") & (test["type"] == "harmful") & (test["attack"] == "plain")
en_b = (test["lang"] == "english") & (test["type"] == "benign")
def proj(l, offset):
    return np.array([float(h[l + offset] @ dirs[l]) for h in H])
raw = proj(layer, OFFSET)

lo, hi = raw[en_b].mean(), raw[en_h].mean()
test["refusal_signal"] = (raw - lo) / (hi - lo)

print("\n=== Mean refusal signal (0 = English harmless, 1 = English harmful plain) ===")
print(test.pivot_table(index=["type", "attack"], columns="lang", values="refusal_signal", aggfunc="mean").round(2))

te_h = (test["lang"] == "telugu") & (test["type"] == "harmful") & (test["attack"] == "plain")
stat, p = mannwhitneyu(test.loc[en_h, "refusal_signal"], test.loc[te_h, "refusal_signal"])
print(f"\nEnglish vs Telugu, plain harmful prompts: Mann-Whitney p = {p:.5f}")

print("\nAUC, refusal signal vs refused/harmful_compliance:")
harm = test[(test["type"] == "harmful") & test["label"].isin(["refused", "harmful_compliance"])]
for lang in ["english", "telugu", None]:
    sub = harm if lang is None else harm[harm["lang"] == lang]
    y = (sub["label"] == "refused").astype(int)
    if y.nunique() == 2:
        print(f"{lang or 'both'}: AUC = {roc_auc_score(y, sub['refusal_signal']):.2f}  (n={len(sub)})")

layers = sorted(dirs)
plt.figure(figsize=(9, 4))
for name, mask in [("English harmful", en_h), ("Telugu harmful", te_h),
                   ("English harmless", en_b), ("Telugu harmless", (test["lang"] == "telugu") & (test["type"] == "benign"))]:
    plt.plot(layers, [proj(l, OFFSET)[mask].mean() for l in layers], marker="o", ms=3, label=name)
plt.axvline(layer, ls="--", c="grey", lw=1)
plt.xlabel("Layer"); plt.ylabel("Projection on refusal direction")
plt.title("Tiny Aya Global: refusal signal, English vs Telugu (plain prompts)")
plt.legend(); plt.tight_layout(); plt.savefig("refusal_signal_by_layer.png", dpi=150); plt.show()

test.drop(columns=["reply"]).to_csv("refusal_signal.csv", index=False, encoding="utf-8-sig")
print("\nSaved refusal_signal.csv and refusal_signal_by_layer.png")